# BuzzSpot - Size-routed three-model ensemble

This notebook creates two controlled test submissions from three already-generated prediction files:

- **Small predicted boxes:** existing YOLO full-frame + sliced ensemble
- **Medium predicted boxes:** RF-DETR Large
- **Large predicted boxes:** YOLO full-frame Model A

COCO size boundaries are used exactly:

- small: area `< 32² = 1,024`
- medium: `1,024 ≤ area < 96² = 9,216`
- large: area `≥ 9,216`

No training, model inference, threshold sweep, score scaling, or class gating is performed here.
The only final-fusion alternatives are class-aware NMS at IoU **0.60** and **0.50**.


## 1. Mount Drive and locate the test-phase annotation archive


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

tar_hits = glob.glob(
    "/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar",
    recursive=True,
)
if not tar_hits:
    tar_hits = glob.glob(
        "/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*",
        recursive=True,
    )

assert tar_hits, "BuzzSpot test-phase archive not found under MyDrive"
TAR_PATH = Path(tar_hits[0])

print("test-phase archive:", TAR_PATH)


Mounted at /content/drive
test-phase archive: /content/drive/MyDrive/Colab Notebooks/CVPPA ECCV 26 BuzzSpot Challenge/19557529600-BuzzSet_challenge_testphase.tar


## 2. Locked ensemble configuration and exact input files


In [2]:
import json
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"
ENSEMBLE_DIR = PROJECT_DIR / "ensemble_size_routed"

SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    1: "bee",
    2: "bumblebee",
    3: "hoverfly",
    4: "moth",
}

# Exact prediction files produced by the completed notebooks.
YOLO_FULL_PRED_PATH = SUBMISSIONS_DIR / (
    "predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.json"
)

YOLO_AB_SMALL_PRED_PATH = SUBMISSIONS_DIR / (
    "predictions_"
    "yolo26m_32ep_rare_os_cm5_trainval_finalfit"
    "_plus_"
    "yolo26m_sliced700_trainval_from_fullframeA_12ep"
    "_AplusB_gated_no_hoverfly_Bnohoverfly"
    "_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.json"
)

RFDETR_PRED_PATH = SUBMISSIONS_DIR / (
    "predictions_rfdetr_large_trainval_os_1120_32ep"
    "_regular_conf010_top100.json"
)

# Exact COCO size boundaries in original-image pixel area.
SMALL_AREA_MAX = 32 ** 2       # 1,024
MEDIUM_AREA_MAX = 96 ** 2      # 9,216

# One primary submission and one conservative alternate.
FINAL_NMS_IOU_VALUES = (0.60, 0.50)
MAX_DET_FINAL = 100

# Scores are left exactly as produced by each existing model pipeline.
SOURCE_SCORE_SCALES = {
    "yolo_ab_small": 1.0,
    "rfdetr_medium": 1.0,
    "yolo_full_large": 1.0,
}

# Hard guards against accidental experimentation.
assert SMALL_AREA_MAX == 1024
assert MEDIUM_AREA_MAX == 9216
assert FINAL_NMS_IOU_VALUES == (0.60, 0.50)
assert MAX_DET_FINAL == 100
assert set(SOURCE_SCORE_SCALES.values()) == {1.0}
assert "conf010_top100" in RFDETR_PRED_PATH.name
assert "regular_conf001.json" not in RFDETR_PRED_PATH.name

for path in [
    YOLO_FULL_PRED_PATH,
    YOLO_AB_SMALL_PRED_PATH,
    RFDETR_PRED_PATH,
]:
    assert path.exists(), f"Missing required prediction file: {path}"

SELECTED_CONFIG = {
    "small_source": YOLO_AB_SMALL_PRED_PATH.name,
    "medium_source": RFDETR_PRED_PATH.name,
    "large_source": YOLO_FULL_PRED_PATH.name,
    "small_area": [0, SMALL_AREA_MAX],
    "medium_area": [SMALL_AREA_MAX, MEDIUM_AREA_MAX],
    "large_area": [MEDIUM_AREA_MAX, None],
    "score_scales": SOURCE_SCORE_SCALES,
    "final_nms_iou_values": list(FINAL_NMS_IOU_VALUES),
    "max_det_per_image": MAX_DET_FINAL,
}

config_path = ENSEMBLE_DIR / "selected_size_routed_config.json"
config_path.write_text(json.dumps(SELECTED_CONFIG, indent=2))

print(json.dumps(SELECTED_CONFIG, indent=2))
print("saved config:", config_path)


{
  "small_source": "predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.json",
  "medium_source": "predictions_rfdetr_large_trainval_os_1120_32ep_regular_conf010_top100.json",
  "large_source": "predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.json",
  "small_area": [
    0,
    1024
  ],
  "medium_area": [
    1024,
    9216
  ],
  "large_area": [
    9216,
    null
  ],
  "score_scales": {
    "yolo_ab_small": 1.0,
    "rfdetr_medium": 1.0,
    "yolo_full_large": 1.0
  },
  "final_nms_iou_values": [
    0.6,
    0.5
  ],
  "max_det_per_image": 100
}
saved config: /content/drive/MyDrive/BuzzSpot/ensemble_size_routed/selected_size_routed_config.json


## 3. Load test keyframe IDs and original image dimensions


In [3]:
import json
import tarfile
from pathlib import Path

tf = tarfile.open(TAR_PATH, "r:*")
tar_members = {
    member.name: member
    for member in tf.getmembers()
    if member.isfile()
}
tar_names = set(tar_members)

def find_tar_member(relative_path):
    relative_path = relative_path.lstrip("/")
    candidates = (
        relative_path,
        f"BuzzSet_challenge/{relative_path}",
        f"BuzzSet_challenge_testphase/{relative_path}",
    )

    for candidate in candidates:
        if candidate in tar_names:
            return candidate

    suffix = "/" + relative_path
    matches = [
        name for name in tar_names
        if name.endswith(suffix)
    ]
    return matches[0] if matches else None

annotation_member = (
    find_tar_member("annotations/test_testphase.json")
    or find_tar_member("test_testphase.json")
)
assert annotation_member is not None, "test_testphase.json not found in archive"

with tf.extractfile(tar_members[annotation_member]) as file:
    test_coco = json.load(file)

tf.close()

keyframe_images = [
    image
    for image in test_coco["images"]
    if image.get("is_keyframe") in [True, 1, "true", "True"]
]

TEST_DIMENSIONS = {
    int(image["id"]): (
        int(image.get("width", 1920)),
        int(image.get("height", 1080)),
    )
    for image in keyframe_images
}
TEST_IMAGE_IDS = set(TEST_DIMENSIONS)

assert len(TEST_IMAGE_IDS) == 4763, (
    f"Expected 4,763 test keyframes, found {len(TEST_IMAGE_IDS)}"
)

print("test keyframes:", len(TEST_IMAGE_IDS))
print("first IDs:", sorted(TEST_IMAGE_IDS)[:5])


test keyframes: 4763
first IDs: [6, 12, 18, 24, 30]


## 4. Load and strictly validate the three input prediction files


In [4]:
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

def load_predictions(path):
    path = Path(path)
    predictions = json.loads(path.read_text())
    assert isinstance(predictions, list), f"{path.name} must contain a JSON list"
    print("loaded:", path)
    return predictions

def bbox_area_xywh(bbox):
    _, _, width, height = map(float, bbox)
    return max(0.0, width) * max(0.0, height)

def area_band_from_bbox(bbox):
    area = bbox_area_xywh(bbox)

    if area < SMALL_AREA_MAX:
        return "small"
    if area < MEDIUM_AREA_MAX:
        return "medium"
    return "large"

def validate_prediction_list(predictions, source_name):
    required_keys = {"image_id", "category_id", "bbox", "score"}

    per_image = Counter()
    class_counts = Counter()
    band_counts = Counter()

    for index, prediction in enumerate(predictions):
        assert required_keys.issubset(prediction), (
            f"{source_name}[{index}] missing required keys"
        )

        image_id = int(prediction["image_id"])
        category_id = int(prediction["category_id"])
        score = float(prediction["score"])
        bbox = prediction["bbox"]

        assert image_id in TEST_IMAGE_IDS, (
            f"{source_name} contains non-keyframe image ID {image_id}"
        )
        assert category_id in CLASS_NAMES, (
            f"{source_name} contains invalid category {category_id}"
        )
        assert math.isfinite(score) and 0.0 <= score <= 1.0, (
            f"{source_name} contains invalid score {score}"
        )
        assert isinstance(bbox, list) and len(bbox) == 4, (
            f"{source_name} contains invalid bbox {bbox}"
        )

        x, y, width, height = map(float, bbox)
        image_width, image_height = TEST_DIMENSIONS[image_id]

        assert all(math.isfinite(value) for value in [x, y, width, height])
        assert width >= 1.0 and height >= 1.0, (
            f"{source_name} contains degenerate bbox {bbox}"
        )
        assert x >= -1e-6 and y >= -1e-6
        assert x + width <= image_width + 1e-6
        assert y + height <= image_height + 1e-6

        per_image[image_id] += 1
        class_counts[category_id] += 1
        band_counts[area_band_from_bbox(bbox)] += 1

    maximum = max(per_image.values(), default=0)
    assert maximum <= 100, (
        f"{source_name} has {maximum} predictions on one image; expected <= 100"
    )

    print(f"\n{source_name}")
    print("  detections:", len(predictions))
    print("  predicted images:", len(per_image), "/", len(TEST_IMAGE_IDS))
    print("  max detections/image:", maximum)
    print("  class counts:", {
        CLASS_NAMES[category_id]: class_counts[category_id]
        for category_id in CLASS_NAMES
    })
    print("  predicted-area bands:", dict(band_counts))

yolo_full_predictions = load_predictions(YOLO_FULL_PRED_PATH)
yolo_ab_predictions = load_predictions(YOLO_AB_SMALL_PRED_PATH)
rfdetr_predictions = load_predictions(RFDETR_PRED_PATH)

validate_prediction_list(yolo_full_predictions, "YOLO full-frame")
validate_prediction_list(yolo_ab_predictions, "YOLO A+B ensemble")
validate_prediction_list(rfdetr_predictions, "RF-DETR")


loaded: /content/drive/MyDrive/BuzzSpot/submissions/predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_conf001.json
loaded: /content/drive/MyDrive/BuzzSpot/submissions/predictions_yolo26m_32ep_rare_os_cm5_trainval_finalfit_plus_yolo26m_sliced700_trainval_from_fullframeA_12ep_AplusB_gated_no_hoverfly_Bnohoverfly_a0p001_b0p001_bm0p5_bscale0p75_fmnms_iou_0p6.json
loaded: /content/drive/MyDrive/BuzzSpot/submissions/predictions_rfdetr_large_trainval_os_1120_32ep_regular_conf010_top100.json

YOLO full-frame
  detections: 45883
  predicted images: 4721 / 4763
  max detections/image: 65
  class counts: {'bee': 26205, 'bumblebee': 1223, 'hoverfly': 17653, 'moth': 802}
  predicted-area bands: {'medium': 23787, 'small': 20571, 'large': 1525}

YOLO A+B ensemble
  detections: 48750
  predicted images: 4741 / 4763
  max detections/image: 65
  class counts: {'bee': 31432, 'bumblebee': 1971, 'hoverfly': 12833, 'moth': 2514}
  predicted-area bands: {'medium': 22041, 'small': 24917, 'large': 1792}



## 5. Geometry, hard size routing, and class-aware NMS


In [5]:
from collections import Counter, defaultdict

def coco_prediction_to_internal(prediction, source):
    x, y, width, height = map(float, prediction["bbox"])

    return {
        "image_id": int(prediction["image_id"]),
        "category_id": int(prediction["category_id"]),
        "bbox_xyxy": [
            x,
            y,
            x + width,
            y + height,
        ],
        "score": float(prediction["score"]),
        "source": source,
    }

def internal_area(prediction):
    x1, y1, x2, y2 = prediction["bbox_xyxy"]
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def intersection_area_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    x1 = max(ax1, bx1)
    y1 = max(ay1, by1)
    x2 = min(ax2, bx2)
    y2 = min(ay2, by2)

    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def iou_xyxy(a, b):
    intersection = intersection_area_xyxy(a, b)

    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - intersection

    return 0.0 if union <= 0 else intersection / union

def class_aware_nms(predictions, iou_threshold, max_detections=100):
    kept = []

    for category_id in sorted({
        int(prediction["category_id"])
        for prediction in predictions
    }):
        class_predictions = [
            prediction
            for prediction in predictions
            if int(prediction["category_id"]) == category_id
        ]
        class_predictions.sort(
            key=lambda prediction: float(prediction["score"]),
            reverse=True,
        )

        while class_predictions:
            top = class_predictions.pop(0)
            kept.append(top)

            class_predictions = [
                candidate
                for candidate in class_predictions
                if iou_xyxy(
                    top["bbox_xyxy"],
                    candidate["bbox_xyxy"],
                ) < iou_threshold
            ]

    kept.sort(
        key=lambda prediction: float(prediction["score"]),
        reverse=True,
    )
    return kept[:max_detections]

def internal_to_coco(prediction):
    x1, y1, x2, y2 = prediction["bbox_xyxy"]

    return {
        "image_id": int(prediction["image_id"]),
        "category_id": int(prediction["category_id"]),
        "bbox": [
            round(float(x1), 2),
            round(float(y1), 2),
            round(float(x2 - x1), 2),
            round(float(y2 - y1), 2),
        ],
        "score": round(float(prediction["score"]), 5),
    }

def group_by_image(predictions, source):
    grouped = defaultdict(list)

    for prediction in predictions:
        internal = coco_prediction_to_internal(prediction, source)
        grouped[internal["image_id"]].append(internal)

    return grouped

yolo_full_by_image = group_by_image(
    yolo_full_predictions,
    "yolo_full_large",
)
yolo_ab_by_image = group_by_image(
    yolo_ab_predictions,
    "yolo_ab_small",
)
rfdetr_by_image = group_by_image(
    rfdetr_predictions,
    "rfdetr_medium",
)

# Boundary behavior matches COCO exactly.
assert area_band_from_bbox([0, 0, 31, 33]) == "small"       # 1,023
assert area_band_from_bbox([0, 0, 32, 32]) == "medium"      # 1,024
assert area_band_from_bbox([0, 0, 95, 97]) == "medium"      # 9,215
assert area_band_from_bbox([0, 0, 96, 96]) == "large"       # 9,216

print("geometry and routing helper checks passed")


geometry and routing helper checks passed


## 6. Compose the hard size-routed candidates


In [6]:
from collections import Counter, defaultdict

routed_candidates_by_image = defaultdict(list)
routed_source_counts = Counter()
routed_class_counts = Counter()

for image_id in sorted(TEST_IMAGE_IDS):
    # Small predicted boxes come only from the established YOLO A+B ensemble.
    for prediction in yolo_ab_by_image.get(image_id, []):
        area = internal_area(prediction)
        if area < SMALL_AREA_MAX:
            routed_candidates_by_image[image_id].append(prediction)
            routed_source_counts[prediction["source"]] += 1
            routed_class_counts[
                (prediction["source"], prediction["category_id"])
            ] += 1

    # Medium predicted boxes come only from RF-DETR Large.
    for prediction in rfdetr_by_image.get(image_id, []):
        area = internal_area(prediction)
        if SMALL_AREA_MAX <= area < MEDIUM_AREA_MAX:
            routed_candidates_by_image[image_id].append(prediction)
            routed_source_counts[prediction["source"]] += 1
            routed_class_counts[
                (prediction["source"], prediction["category_id"])
            ] += 1

    # Large predicted boxes come only from YOLO full-frame Model A.
    for prediction in yolo_full_by_image.get(image_id, []):
        area = internal_area(prediction)
        if area >= MEDIUM_AREA_MAX:
            routed_candidates_by_image[image_id].append(prediction)
            routed_source_counts[prediction["source"]] += 1
            routed_class_counts[
                (prediction["source"], prediction["category_id"])
            ] += 1

# Hard provenance checks: no source may leak into the wrong size band.
for image_predictions in routed_candidates_by_image.values():
    for prediction in image_predictions:
        area = internal_area(prediction)
        source = prediction["source"]

        if source == "yolo_ab_small":
            assert area < SMALL_AREA_MAX
        elif source == "rfdetr_medium":
            assert SMALL_AREA_MAX <= area < MEDIUM_AREA_MAX
        elif source == "yolo_full_large":
            assert area >= MEDIUM_AREA_MAX
        else:
            raise AssertionError(f"Unknown source: {source}")

print("routed candidate counts:", dict(routed_source_counts))
print("total routed candidates:", sum(routed_source_counts.values()))

for source in SOURCE_SCORE_SCALES:
    print(f"\n{source} class counts:")
    print({
        CLASS_NAMES[category_id]: routed_class_counts[(source, category_id)]
        for category_id in CLASS_NAMES
    })


routed candidate counts: {'yolo_ab_small': 24917, 'rfdetr_medium': 63895, 'yolo_full_large': 1525}
total routed candidates: 90337

yolo_ab_small class counts:
{'bee': 15823, 'bumblebee': 642, 'hoverfly': 8251, 'moth': 201}

rfdetr_medium class counts:
{'bee': 37561, 'bumblebee': 7430, 'hoverfly': 8376, 'moth': 10528}

yolo_full_large class counts:
{'bee': 1046, 'bumblebee': 130, 'hoverfly': 153, 'moth': 196}


## 7. Create primary NMS 0.60 and alternate NMS 0.50 submissions


In [7]:
import json
import shutil
import zipfile
from collections import Counter
from pathlib import Path

OUTPUT_TAG = (
    "size_routed_"
    "yoloAB_small_"
    "rfdetr_medium_"
    "yoloA_large"
)

generated_outputs = {}

for final_nms_iou in FINAL_NMS_IOU_VALUES:
    final_internal = []
    pre_nms_count = 0

    for image_id in sorted(TEST_IMAGE_IDS):
        candidates = routed_candidates_by_image.get(image_id, [])
        pre_nms_count += len(candidates)

        merged = class_aware_nms(
            candidates,
            iou_threshold=final_nms_iou,
            max_detections=MAX_DET_FINAL,
        )
        final_internal.extend(merged)

    final_predictions = [
        internal_to_coco(prediction)
        for prediction in final_internal
    ]

    nms_tag = f"nms{int(round(final_nms_iou * 100)):03d}"
    local_prediction_path = Path(
        f"/content/predictions_{OUTPUT_TAG}_{nms_tag}_top100.json"
    )
    local_zip_path = Path(
        f"/content/submission_{OUTPUT_TAG}_{nms_tag}_top100.zip"
    )

    drive_prediction_path = SUBMISSIONS_DIR / local_prediction_path.name
    drive_zip_path = SUBMISSIONS_DIR / local_zip_path.name

    local_prediction_path.write_text(json.dumps(final_predictions))

    with zipfile.ZipFile(
        local_zip_path,
        "w",
        zipfile.ZIP_DEFLATED,
    ) as archive:
        archive.write(
            local_prediction_path,
            arcname="predictions.json",
        )

    shutil.copy2(local_prediction_path, drive_prediction_path)
    shutil.copy2(local_zip_path, drive_zip_path)

    source_counts_after_nms = Counter(
        prediction["source"]
        for prediction in final_internal
    )
    class_counts = Counter(
        int(prediction["category_id"])
        for prediction in final_internal
    )
    per_image_counts = Counter(
        int(prediction["image_id"])
        for prediction in final_internal
    )

    generated_outputs[final_nms_iou] = {
        "predictions": final_predictions,
        "internal": final_internal,
        "local_prediction_path": local_prediction_path,
        "local_zip_path": local_zip_path,
        "drive_prediction_path": drive_prediction_path,
        "drive_zip_path": drive_zip_path,
    }

    print(f"\nFinal NMS IoU: {final_nms_iou:.2f}")
    print("  candidates before final NMS:", pre_nms_count)
    print("  detections after final NMS:", len(final_predictions))
    print("  suppressed:", pre_nms_count - len(final_predictions))
    print("  source counts after NMS:", dict(source_counts_after_nms))
    print("  class counts:", {
        CLASS_NAMES[category_id]: class_counts[category_id]
        for category_id in CLASS_NAMES
    })
    print("  predicted images:", len(per_image_counts), "/", len(TEST_IMAGE_IDS))
    print("  max detections/image:", max(per_image_counts.values(), default=0))
    print("  Drive zip:", drive_zip_path)



Final NMS IoU: 0.60
  candidates before final NMS: 90337
  detections after final NMS: 77429
  suppressed: 12908
  source counts after NMS: {'rfdetr_medium': 51956, 'yolo_ab_small': 24370, 'yolo_full_large': 1103}
  class counts: {'bee': 45551, 'bumblebee': 7253, 'hoverfly': 15827, 'moth': 8798}
  predicted images: 4756 / 4763
  max detections/image: 94
  Drive zip: /content/drive/MyDrive/BuzzSpot/submissions/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms060_top100.zip

Final NMS IoU: 0.50
  candidates before final NMS: 90337
  detections after final NMS: 71741
  suppressed: 18596
  source counts after NMS: {'rfdetr_medium': 47852, 'yolo_ab_small': 22891, 'yolo_full_large': 998}
  class counts: {'bee': 41445, 'bumblebee': 6974, 'hoverfly': 15054, 'moth': 8268}
  predicted images: 4756 / 4763
  max detections/image: 82
  Drive zip: /content/drive/MyDrive/BuzzSpot/submissions/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms050_top100.zip


## 8. Strictly validate both submission archives


In [8]:
import json
import math
import zipfile
from collections import Counter

def validate_final_submission(prediction_path, zip_path):
    predictions = json.loads(Path(prediction_path).read_text())

    with zipfile.ZipFile(zip_path, "r") as archive:
        names = archive.namelist()

    assert names == ["predictions.json"], (
        f"Zip must contain exactly predictions.json, found {names}"
    )

    per_image = Counter()
    exact_records = set()

    for index, prediction in enumerate(predictions):
        image_id = int(prediction["image_id"])
        category_id = int(prediction["category_id"])
        score = float(prediction["score"])
        x, y, width, height = map(float, prediction["bbox"])

        assert image_id in TEST_IMAGE_IDS
        assert category_id in CLASS_NAMES
        assert math.isfinite(score) and 0.0 <= score <= 1.0
        assert width >= 1.0 and height >= 1.0

        image_width, image_height = TEST_DIMENSIONS[image_id]
        assert x >= -1e-6 and y >= -1e-6
        assert x + width <= image_width + 1e-6
        assert y + height <= image_height + 1e-6

        per_image[image_id] += 1

        exact_record = (
            image_id,
            category_id,
            round(x, 2),
            round(y, 2),
            round(width, 2),
            round(height, 2),
            round(score, 5),
        )
        assert exact_record not in exact_records, (
            f"Exact duplicate prediction found at index {index}"
        )
        exact_records.add(exact_record)

    assert max(per_image.values(), default=0) <= MAX_DET_FINAL

    print("\nvalidation passed:", zip_path)
    print("  detections:", len(predictions))
    print("  predicted images:", len(per_image), "/", len(TEST_IMAGE_IDS))
    print("  zero-prediction images:", len(TEST_IMAGE_IDS - set(per_image)))
    print("  max detections/image:", max(per_image.values(), default=0))
    print("  zip contents:", names)

for final_nms_iou, output in generated_outputs.items():
    validate_final_submission(
        output["local_prediction_path"],
        output["local_zip_path"],
    )



validation passed: /content/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms060_top100.zip
  detections: 77429
  predicted images: 4756 / 4763
  zero-prediction images: 7
  max detections/image: 94
  zip contents: ['predictions.json']

validation passed: /content/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms050_top100.zip
  detections: 71741
  predicted images: 4756 / 4763
  zero-prediction images: 7
  max detections/image: 82
  zip contents: ['predictions.json']


## 9. Final output summary


In [9]:
print("Primary submission — final class-aware NMS IoU 0.60:")
print(generated_outputs[0.60]["drive_zip_path"])

print("\nAlternate submission — final class-aware NMS IoU 0.50:")
print(generated_outputs[0.50]["drive_zip_path"])

print("\nSubmit the 0.60 version first.")
print(
    "The 0.50 version changes only final duplicate suppression; "
    "all model outputs, score scales, size boundaries, and top-100 behavior are identical."
)


Primary submission — final class-aware NMS IoU 0.60:
/content/drive/MyDrive/BuzzSpot/submissions/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms060_top100.zip

Alternate submission — final class-aware NMS IoU 0.50:
/content/drive/MyDrive/BuzzSpot/submissions/submission_size_routed_yoloAB_small_rfdetr_medium_yoloA_large_nms050_top100.zip

Submit the 0.60 version first.
The 0.50 version changes only final duplicate suppression; all model outputs, score scales, size boundaries, and top-100 behavior are identical.
